# 🌲 Week 11 Lab — SOLUTIONS
## Random Forests & Ensemble Methods

**Objectives:**
- Build a single decision tree and observe overfitting
- Train a random forest and understand how averaging reduces error
- Compare RF with the six methods from Weeks 5–10
- Use permutation importance to identify which neurons and muscles matter
- Compare RF with gradient boosting
- Evaluate robustness to simulated electrode drift

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import time
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, LeaveOneGroupOut, StratifiedKFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})
logo = LeaveOneGroupOut()

## Upload Data

Upload `week8_data.pkl` — the same reaching dataset from Weeks 8–10.

This file contains:
- 480 trials (8 directions × 3 speeds × 20 subjects)
- 80 neural features (cosine-tuned M1 neurons)
- 6 EMG features (muscle activations)
- Subject labels for leave-one-subject-out cross-validation

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload week8_data.pkl

In [ ]:
# Load data
with open('week8_data.pkl', 'rb') as f:
    D = pickle.load(f)

X_neural = D['neural_rates']   # (480, 80)
X_emg    = D['X_raw']          # (480, 6)
y_dir    = D['targets']        # 8 directions (0-7)
y_bin    = (D['labels'] == 'impaired').astype(int)  # 0=healthy, 1=impaired
subjects = D['subjects']       # 20 subjects

muscle_names = ['BIC', 'TRI', 'AD', 'PD', 'BRD', 'PRO']

print(f'Dataset: {X_neural.shape[0]} trials, {X_neural.shape[1]} neurons, '
      f'{X_emg.shape[1]} muscles, {len(np.unique(subjects))} subjects')
print(f'Tasks: {len(np.unique(y_dir))} directions, '
      f'{np.sum(y_bin==0)} healthy + {np.sum(y_bin==1)} impaired')

---
## Part 1: Decision Trees 🟢

We start with the building block of random forests: a single decision tree.

### Exercise 1.1: Visualise a decision tree

Train a decision tree (depth=3) on the 6 muscle features for the healthy-vs-impaired task.
Use `plot_tree` to visualise the tree structure.

**What to look for:** Which muscle does the tree split on first? This is the feature the tree considers most informative.

In [ ]:
# Exercise 1.1: Visualise a decision tree
sc_emg = StandardScaler().fit_transform(X_emg)

# TODO: Train a DecisionTreeClassifier with max_depth=3 on sc_emg, y_bin
# TODO: Use plot_tree() to visualise it
# Hint: use feature_names=muscle_names, class_names=['Healthy', 'Impaired']
# YOUR CODE HERE

### Exercise 1.1b: Decision boundary vs LDA

Project the EMG data to 2D (PCA) and plot the tree's decision boundary alongside LDA's boundary from Week 10.

**What to look for:** The tree produces axis-aligned rectangles; LDA produces a smooth diagonal line. Both separate the classes, but in very different ways.

In [ ]:
# Exercise 1.1b: Decision boundary in PCA space
from matplotlib.colors import ListedColormap

pca2 = PCA(n_components=2).fit(sc_emg)
X_pca = pca2.transform(sc_emg)

# TODO: Train a DecisionTreeClassifier (depth=4) and LDA on X_pca, y_bin
# TODO: Create a meshgrid over the PCA space
# TODO: Plot tree boundary as filled regions (contourf) and LDA as dashed contour
# TODO: Overlay the data points coloured by class
# YOUR CODE HERE

### Exercise 1.2: Overfitting with depth

Train decision trees with depth 1 to 20 on **both** the EMG healthy-vs-impaired task and the neural 8-direction task. Plot training vs LOSO test accuracy side by side.

**Expected result:** The EMG task (Panel A) never overfits — with only 6 features and 2 classes, there is not enough room for the tree to memorise noise. The neural task (Panel B) shows the classic overfitting gap — 80 noisy features and 8 classes give the tree plenty of room to memorise subject-specific patterns.

In [ ]:
# Exercise 1.2: Overfitting with depth — two tasks side by side
depths = list(range(1, 21))

# TODO: Create a 1x2 figure
# Panel A: EMG healthy-vs-impaired (X_emg, y_bin)
# Panel B: Neural 8-direction (X_neural, y_dir)
# For each panel, plot training accuracy and LOSO test accuracy vs depth
# YOUR CODE HERE

---
## Part 2: From Trees to Forests 🟢

Now we combine many trees into a random forest and see how averaging eliminates the overfitting problem from Part 1.

### Exercise 2.1: Accuracy vs number of trees

Train random forests with 1, 2, 5, 10, 20, 50, 100, and 200 trees on the neural 8-direction task (LOSO). Plot accuracy vs number of trees.

**Expected result:** Accuracy climbs steeply with the first few trees, then plateaus. Unlike depth (Exercise 1.2), more trees never hurts.

In [ ]:
# Exercise 2.1: Accuracy vs number of trees
n_trees_list = [1, 2, 5, 10, 20, 50, 100, 200]
tree_accs = []

# TODO: For each n in n_trees_list, train a RandomForestClassifier
#   and compute LOSO accuracy
# TODO: Plot accuracy vs n_trees (semilog x-axis)
# YOUR CODE HERE

### Exercise 2.1b: Visualising tree disagreement

Project the neural data to 2D (PCA) and compare how **three individual bootstrapped trees** disagree vs how the **200-tree forest** produces a stable consensus.

**What to look for:** In Panel A, the darkened regions show where the three trees predict different directions — they disagree on a large fraction of the space. In Panel B, the forest's majority vote is stable almost everywhere, with uncertainty confined to narrow boundary bands.

In [ ]:
# Exercise 2.1b: Visualising tree disagreement vs forest consensus
X_pca = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X_neural))

# TODO: Create a meshgrid over PCA space
# TODO Panel A: Train 3 trees on different bootstrap samples, predict on grid,
#   shade regions where they disagree
# TODO Panel B: Train a 200-tree forest, predict on grid,
#   shade only low-confidence regions (predict_proba < 0.4)
# YOUR CODE HERE

### Exercise 2.2: OOB vs LOSO

Random forests give a free accuracy estimate via out-of-bag (OOB) scoring.
Compare OOB accuracy with LOSO accuracy across all four tasks.

**Why it works:** Each tree is trained on a bootstrap sample (~63% of trials). The remaining ~37% were never seen by that tree, so the forest can predict each trial using only the trees that didn't train on it — no extra models needed.

In [ ]:
# Exercise 2.2: OOB vs LOSO
tasks = {
    'EMG Dir.': (X_emg, y_dir),
    'Neural Dir.': (X_neural, y_dir),
    'EMG Bin.': (X_emg, y_bin),
    'Neural Bin.': (X_neural, y_bin),
}

# TODO: For each task, compute:
#   1. OOB accuracy: fit RF with oob_score=True, read rf.oob_score_
#   2. LOSO accuracy: cross_val_score with logo
# TODO: Plot side-by-side bars
# YOUR CODE HERE

---
## Part 3: Seven-Method Comparison 🟡

Add RF to the six-method comparison table from Week 10.

### Exercise 3.1: LOSO comparison

Evaluate all seven methods on all four tasks using LOSO.
Plot the results as a grouped bar chart.

**Expected result:** RF is competitive with the best method on every task without any task-specific tuning.

In [ ]:
# Exercise 3.1: Seven-method LOSO comparison
methods = {
    'NB': GaussianNB(),
    'LR': LogisticRegression(C=10, max_iter=2000),
    'KNN': KNeighborsClassifier(5),
    'Lin SVM': SVC(kernel='linear', C=1),
    'RBF SVM': SVC(kernel='rbf', C=10, gamma='scale'),
    'LDA': LinearDiscriminantAnalysis(),
    'RF': RandomForestClassifier(n_estimators=200, random_state=42),
}

# TODO: Evaluate all methods on all 4 tasks using LOSO
# TODO: Plot grouped bar charts (direction decoding + binary diagnosis)
# YOUR CODE HERE

### Exercise 3.2: Hyperparameter sensitivity

RF has four main hyperparameters: `n_estimators` (number of trees), `max_depth` (how deep each tree grows), `max_features` (how many neurons each split considers), and `min_samples_leaf` (smallest leaf size).

Sweep each one across a range while holding the others at their defaults. Plot a 2×2 grid.

**Expected result:** All four curves are remarkably flat — RF's defaults work well without tuning. Compare this with the sharp sensitivity to C and γ in SVM (Week 9).

In [ ]:
# Exercise 3.2: Hyperparameter sensitivity (neural 8-direction, 5-fold CV)
skf = StratifiedKFold(5, shuffle=True, random_state=42)

# TODO: Create a 2x2 figure
# (A) Sweep n_estimators: [1, 5, 10, 50, 100, 200, 500]
# (B) Sweep max_depth: [2, 4, 7, 10, 15, 20, None]
# (C) Sweep max_features: [1, 3, 5, 9, 20, 40, 80]
# (D) Sweep min_samples_leaf: [1, 2, 5, 10, 15, 20]
# Use 5-fold CV (skf) for speed
# YOUR CODE HERE

---
## Part 4: Feature Importance 🟡

RF's unique strength: telling us *which* features drive the classification.

**Important:** Permutation importance must be computed on held-out test data. If computed on training data, the forest's perfect memorisation makes every feature appear unimportant.

### Exercise 4.1: Muscle importance for diagnosis

Use permutation importance to rank the 6 muscles for the healthy-vs-impaired task.
Compare with LDA Fisher weights, LR coefficients, and PCA loadings.

**Question to consider:** Do the four methods agree? If not, why might they disagree?

In [ ]:
# Exercise 4.1: Muscle importance — four methods compared
from sklearn.model_selection import train_test_split

# TODO: Split sc_emg into train/test (test_size=0.3, stratify=y_bin)
# TODO: Train RF on train set, compute permutation_importance on test set
# TODO: Fit LDA, LR, PCA on full data for comparison weights
# TODO: Create 2x2 bar chart comparing all four methods
# YOUR CODE HERE

### Exercise 4.2: Neural importance

Compute permutation importance for all 80 neurons on the 8-direction task.
Colour each bar by the neuron's preferred direction.

**Expected result:** No single neuron dominates — importance is distributed across many neurons, consistent with the cosine tuning model from Week 6.

In [ ]:
# Exercise 4.2: Neural importance for 8-direction decoding
# TODO: Train/test split on standardised neural data
# TODO: Train RF, compute permutation_importance on test set
# TODO: Colour bars by preferred direction
# Hint: preferred direction = direction with highest mean firing rate
# YOUR CODE HERE

---
## Part 5: Gradient Boosting 🔴

Compare the two ensemble strategies: RF (parallel averaging) vs gradient boosting (sequential error correction).

### Exercise 5.1: RF vs Gradient Boosting

Evaluate both ensemble methods on all four tasks using 5-fold CV.

**Expected result:** Similar accuracy on this dataset. Boosting would show a larger advantage on data with complex, nonlinear class boundaries.

In [ ]:
# Exercise 5.1: RF vs Gradient Boosting
skf = StratifiedKFold(5, shuffle=True, random_state=42)

# TODO: Evaluate RF and GradientBoostingClassifier on all 4 tasks
# Use 5-fold CV (skf) since LOSO is slow for boosting
# TODO: Plot side-by-side bars
# YOUR CODE HERE

---
## Part 6: Electrode Drift Robustness 🔴

Simulate electrode drift by adding Gaussian noise to the neural features and measure how each method degrades.

### Exercise 6.1: Drift simulation

Add Gaussian noise with σ = 0, 1, 2, 3, 5, 7, 10 to the neural features.
Evaluate all seven methods at each noise level (LOSO).
Plot accuracy vs drift for all methods.

**Question:** Which method is most robust? Which degrades fastest? Why?

In [ ]:
# Exercise 6.1: Electrode drift robustness
drift_levels = [0, 1, 2, 3, 5, 7, 10]

# TODO: For each sigma, add Gaussian noise to X_neural
#   X_drift = X_neural + np.random.randn(*X_neural.shape) * sigma
# TODO: Evaluate all 7 methods with LOSO
# TODO: Plot accuracy vs drift for all methods
# YOUR CODE HERE

---
## 💭 Thought Exercise

A rehabilitation clinic wants to build a BCI decoder for 8-direction reaching.
They have 480 trials from 20 patients and 80-channel neural recordings.

1. Based on the results above, which method would you recommend as the **first thing to try**? Why?

2. The clinic also wants to know **which neurons are most important** for the decoder, so they can prioritise those channels during surgery. Which method provides this information? What caveat would you mention about interpreting the importance rankings?

3. After 6 months, some electrodes have drifted. Based on Figure 6.1, how much accuracy loss would you expect? Would you recommend recalibrating the decoder or switching to a different method?